# Lesson 11: Chaining Agents and Pipelines

## What is Agent Chaining?

**Agent chaining** is when the output of **one agent** becomes the input for **the next agent**.

Like an assembly line:
- Worker 1 does step 1, passes it to worker 2
- Worker 2 does step 2, passes it to worker 3

In AI:
- Researcher Agent **finds information** and passes it to Writer Agent
- Writer Agent **writes an article** from that info and passes it to Editor Agent

Each agent does **one thing well**, then hands off to the next.

> **Cost:** ~$0.05-0.10 (2 chained Sonnet API calls). Takes 30-60 seconds. Run each cell once.

## Plan: 2 Chained Agents

We'll build:

```
[Agent 1: Researcher]  →  research notes  →  [Agent 2: Writer]
    Searches the web                           Writes from research
```

- **Agent 1 (Researcher):** Has DataForSEO search tool (same tool from Lesson 9)
- **Agent 2 (Writer):** Receives research results, writes a paragraph

Chaining is straightforward:
```python
research = researcher.run("Search for info...")
article = writer.run(f"Write from: {research.content}")  # Pass output as input
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

# Set up search tools (same as Lesson 9)
creds = get_dataforseo_credentials()
search_tools = []
if creds:
    search_tools = [DataForSEOSearchTools(login=creds[0], password=creds[1])]
    print("DataForSEO search enabled!")
else:
    print("No DataForSEO key — researcher will use built-in knowledge only.")

# Agent 1: Researcher — searches the web
researcher = Agent(
    name="Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web and return detailed research notes."],
)

# Agent 2: Writer — writes from research
writer = Agent(
    name="Writer",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "Write a short paragraph (3-4 sentences) based on the research notes provided.",
        "Write clearly and professionally.",
    ],
)

print("Created 2 agents: Researcher and Writer")

In [ ]:
# Step 1: Research
print("Step 1: Researching...")
research = researcher.run("Find information about AI trends in SEO for 2026")
print(f"Research done! ({len(research.content)} chars)\n")

# Step 2: Write from research
print("Step 2: Writing...")
article = writer.run(f"Write a paragraph from these research notes:\n\n{research.content}")
print(f"Article:\n{article.content}")

## What Happened

The flow:

1. **Researcher** received the question, searched the web via DataForSEO, and returned `research.content`
2. `research.content` was **passed into** the Writer's prompt
3. **Writer** received the research and wrote a polished paragraph

**The key line:**
```python
article = writer.run(f"Write a paragraph from these research notes:\n\n{research.content}")
```

Pass `research.content` (the previous agent's output) into `writer.run()` — that's it.

## How the Real Product Compares

Our `agentic-content-seo` product uses a variation of this pattern:

- **Content Writer** — Researches AND writes in one agent (chains search → writing internally)
- **Image Finder** — Reads the written article, finds images, inserts them, saves the updated article
- **AIO Analyzer** — Reads the article and analyzes it against Google AI Overviews

Instead of chaining 4 separate agents, the product combines research and writing into **one capable agent** (Content Writer). Then Image Finder and AIO Analyzer each enhance the article.

The **chaining principle** still applies — Image Finder reads what Content Writer wrote, just like your Writer read what Researcher found.

In [ ]:
# Real product flow
print("""
Real product flow:

  [Content Writer] ──article──> [Image Finder] ──enhanced article──> [AIO Analyzer]
       │                              │                                    │
  Search + Write              Find & insert images              Analyze AI Overviews
""")

## Part 2: From Toy Agents to Real Pipelines

The 2-agent chain above is simple: plain text in, plain text out. But real products need **structured data** flowing between agents.

In Lesson 10, you learned that `output_schema` forces an agent to return structured data. When you combine this with chaining, you get a **pipeline** — agents passing structured data to each other.

### Building Nested Schemas

A flat schema like `list[str]` works for simple outlines. But what if each section needs a heading, key points, AND a summary? You need **nested schemas**:

```python
class Section(BaseModel):          # One section
    heading: str
    key_points: list[str]

class DetailedOutline(BaseModel):  # Full outline with sections
    title: str
    sections: list[Section]        # A LIST of Section objects
```

This is the same concept as a dict inside a list inside a dict — but Pydantic **validates** the structure automatically.

In [ ]:
## Exercise — Part 1: Chaining

Add a **third agent** to the chain: an Editor that improves the writer's output.

1. Create an `editor` agent with instructions like: "You are a content editor. Improve the given text: fix grammar, make it more engaging, and ensure it's SEO-friendly. Return only the improved text."
2. Chain it: pass `article.content` into `editor.run()`
3. Print both the original and edited versions

## Exercise

Add a **third agent** to the chain: an Editor that improves the writer's output.

1. Create an `editor` agent with instructions like: "You are a content editor. Improve the given text: fix grammar, make it more engaging, and ensure it's SEO-friendly. Return only the improved text."
2. Chain it: pass `article.content` into `editor.run()`
3. Print both the original and edited versions

This is the same pattern our real pipeline uses — with more agents.

In [ ]:
# Exercise: Fill in the blanks to add a third agent to the chain

from agno.agent import Agent
from agno.models.anthropic import Claude

# The editor agent — improves the writer's output
editor = Agent(
    name=___,                                    # Give it a name
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[___],                          # Tell it to edit and improve text
)

# Chain it: pass the writer's output to the editor
# (assumes `article` variable exists from the cells above)
edited = editor.run(f"Improve this text:\n\n{___}")      # Pass the writer's output

print("=== ORIGINAL ===")
print(article.content)
print("\n=== EDITED ===")
print(___)

## Lesson 11 Summary

### Chaining Agents:
- **Agent chaining**: output of one agent becomes input for the next
- How to do it: pass `response.content` into `agent.run()`
- Each agent only needs to do **one job well**

### Nested Schemas for Pipelines:
- **Nested Pydantic models** let you pass structured data between agents
- `Section` inside `DetailedOutline` — each section has a heading and key points
- Real products combine chaining + structured output for reliable pipelines

### Module 3 — Building Agents:

| Lesson | Concept |
|--------|--------|
| Lesson 8 | Create a basic agent with `name`, `model`, `instructions` |
| Lesson 9 | Add **tools** so agents can act (web search + API concepts) |
| Lesson 10 | **Structured output** — force agents to return formatted data |
| Lesson 11 | **Chain agents** + **nested schemas** — build pipelines |

**Next lesson:** Storage — how articles are saved locally so your agents' work persists.

---

## Ready for Module 4?

After the next lesson (Storage), you'll move to Module 4 where you'll learn **AI-assisted development** — using Claude Code to build the real product.

Three questions to check your understanding:
1. If an agent's `response.content` is plain text, how do you pass it to the next agent?
2. What's the difference between `list[str]` and `list[Section]` as a schema?
3. Why does the real product use one Content Writer instead of separate Researcher + Writer agents?

<details>
<summary>Click to reveal answer</summary>

```python
from agno.agent import Agent
from agno.models.anthropic import Claude

editor = Agent(
    name="Editor",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "You are a content editor. Improve the given text: fix grammar, make it more engaging, and ensure it's SEO-friendly.",
        "Return only the improved text.",
    ],
)

edited = editor.run(f"Improve this text:\n\n{article.content}")

print("=== ORIGINAL ===")
print(article.content)
print("\n=== EDITED ===")
print(edited.content)
```
</details>